# Task 4 – Deployment test
The Flask app (`app/app.py`) serves the best models with two routes:
- `POST /predict/price` → best regression model
- `POST /predict/class?target=gear_type` (or `options`) → best classification model

**Before running this notebook**, in a terminal inside the `project` folder:
```
docker build -t car-api .
docker run -d --name car-api -p 5000:5000 car-api
```
Then run the cells below. At the end, show the container logs with `docker logs car-api` in the terminal.

### (Only if you re-ran notebooks 02 and 03) update requirements.txt
The saved models must be loaded with the same scikit-learn version that created them. This cell writes your
installed versions into `requirements.txt`; then rebuild the image (`docker build -t car-api .`).

In [1]:
import importlib.metadata as md
from pathlib import Path
pkgs = ["numpy", "pandas", "scikit-learn", "joblib", "flask", "matplotlib", "seaborn", "requests"]
lines = [f"{p}=={md.version(p)}" for p in pkgs]
Path("../requirements.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

numpy==2.4.6
pandas==3.0.3
scikit-learn==1.9.0
joblib==1.5.3
flask==3.1.3
matplotlib==3.11.0
seaborn==0.13.2
requests==2.34.2


## Call the API

In [2]:
import json
import requests

BASE = "http://localhost:5000"

def show(title, r):
    print(f"--- {title}  [HTTP {r.status_code}]")
    print(json.dumps(r.json(), indent=2))

show("GET /", requests.get(f"{BASE}/"))

--- GET /  [HTTP 200]
{
  "classification_models": {
    "gear_type": "Random Forest",
    "options": "Random Forest"
  },
  "regression_model": "SVR",
  "status": "ok"
}


In [3]:
cars = [
    {"Make": "Toyota", "Type": "Camry", "Year": 2019, "Origin": "Saudi", "Color": "White",
     "Options": "Semi Full", "Engine_Size": 2.5, "Fuel_Type": "Gas", "Gear_Type": "Automatic",
     "Mileage": 85000, "Region": "Riyadh"},
    {"Make": "Toyota", "Type": "Land Cruiser", "Year": 2017, "Origin": "Gulf Arabic", "Color": "Black",
     "Options": "Full", "Engine_Size": 5.7, "Fuel_Type": "Gas", "Gear_Type": "Automatic",
     "Mileage": 160000, "Region": "Jeddah"},
    {"Make": "Hyundai", "Type": "Accent", "Year": 2012, "Origin": "Saudi", "Color": "Silver",
     "Options": "Standard", "Engine_Size": 1.6, "Fuel_Type": "Gas", "Gear_Type": "Manual",
     "Mileage": 240000, "Region": "Dammam"},
]
show("POST /predict/price", requests.post(f"{BASE}/predict/price", json=cars))

--- POST /predict/price  [HTTP 200]
{
  "model": "SVR",
  "predictions": [
    {
      "predicted_price_SAR": 72500.0
    },
    {
      "predicted_price_SAR": 194100.0
    },
    {
      "predicted_price_SAR": 14900.0
    }
  ]
}


In [4]:
# Classification: the target column is not sent; Price is optional
no_gear = [{k: v for k, v in c.items() if k != "Gear_Type"} | {"Price": p}
           for c, p in zip(cars, [70000, 150000, None])]
show("POST /predict/class?target=gear_type", requests.post(f"{BASE}/predict/class?target=gear_type", json=no_gear))

--- POST /predict/class?target=gear_type  [HTTP 200]
{
  "model": "Random Forest",
  "predictions": [
    {
      "predicted_class": "Automatic",
      "probabilities": {
        "Automatic": 0.967,
        "Manual": 0.033
      }
    },
    {
      "predicted_class": "Automatic",
      "probabilities": {
        "Automatic": 0.97,
        "Manual": 0.03
      }
    },
    {
      "predicted_class": "Automatic",
      "probabilities": {
        "Automatic": 0.7,
        "Manual": 0.3
      }
    }
  ],
  "target": "gear_type"
}


In [5]:
no_opt = [{k: v for k, v in c.items() if k != "Options"} for c in cars]
show("POST /predict/class?target=options", requests.post(f"{BASE}/predict/class?target=options", json=no_opt))

--- POST /predict/class?target=options  [HTTP 200]
{
  "model": "Random Forest",
  "predictions": [
    {
      "predicted_class": "Standard",
      "probabilities": {
        "Full": 0.29,
        "Semi Full": 0.297,
        "Standard": 0.413
      }
    },
    {
      "predicted_class": "Full",
      "probabilities": {
        "Full": 0.726,
        "Semi Full": 0.215,
        "Standard": 0.059
      }
    },
    {
      "predicted_class": "Standard",
      "probabilities": {
        "Full": 0.105,
        "Semi Full": 0.21,
        "Standard": 0.686
      }
    }
  ],
  "target": "options"
}


In [6]:
# A bad request is rejected with HTTP 400
show("Missing Make/Type", requests.post(f"{BASE}/predict/price", json={"Year": 2018}))

--- Missing Make/Type  [HTTP 400]
{
  "error": "Record 0: missing required fields ['Make', 'Type']"
}


Now run `docker logs car-api` in the terminal: every request above appears in the container logs.
When finished: `docker stop car-api` and `docker rm car-api`.